<a href="https://colab.research.google.com/github/Shahad-Hossain/nasa-image-enhancement/blob/main/Homework_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow

In [ ]:
!pip install ipympl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 47.3 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image

In [ ]:
# 1. Pre-process with extra contrast for Martian dust
train_datagen = ImageDataGenerator(
    rescale=1./255,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True
)

In [ ]:
# 2. Load your Mars dataset
train_generator = train_datagen.flow_from_directory(
    'mars_data',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary', # e.g., Rock vs. Not Rock
    classes=['rocks', 'sand']
)

Found 10 images belonging to 2 classes.


In [ ]:
# 3. Use a pre-trained "base"
base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False)

In [ ]:
from skimage.feature import graycomatrix, graycoprops
import numpy as np

In [ ]:
input = "mars_data/sand/sand7.jpg"
try:
    img = Image.open(input).convert('L') # Convert to RGB to ensure consistent 3-channel access
except FileNotFoundError:
    print(f"Error: The file '{input}' was not found.")

pixels = img.load()

width, height = img.size

print(str(width) + " " + str(height))

img.save("grayscaled.jpg")

100 100


In [ ]:
img_array = np.array(img)
img_32 = (img_array / 32).astype(np.uint8)

# 'image' would be your grayscale Mars rock image
# 1. Create the GLCM (The matrix of neighbor relationships)
glcm = graycomatrix(img_32, distances=[5], angles=[0], levels=32,
                        symmetric=True, normed=True)

# 2. Extract the Haralick features
contrast = graycoprops(glcm, 'contrast')[0, 0]
homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
energy = graycoprops(glcm, 'energy')[0, 0]

print(f"Contrast: {contrast}, Homogeneity: {homogeneity}, Energy: {energy}")

Contrast: 0.5672631578947369, Homogeneity: 0.7540105263157895, Energy: 0.4922439320526555


In [ ]:

import matplotlib.pyplot as plt

In [ ]:
# --- 1. Your Existing Data (Contrast and Homogeneity) ---
rocks_x = [0.6965263157894737, 0.46247040802116696, 0.34945054945054943, 0.2659377316530763, 0.6124686716791979]
rocks_y = [0.776180185758514, 0.8267227934954042, 0.8566593406593406, 0.8768161601186063, 0.7510506834312672]

sand_x = [0.23267071124306182, 0.19218679838347552, 0.16777054333183655, 0.0275546422566785, 0.2084650509308044]
sand_y = [0.8899380405318188, 0.9043780871127076, 0.919280422092501, 0.9865589389127591, 0.9073586231120481]

# --- 2. Add Energy Values (Z-axis) ---
# Replace these with your actual Haralick Energy calculations!
rocks_z = [0.15, 0.18, 0.22, 0.25, 0.12]
sand_z = [0.45, 0.52, 0.58, 0.75, 0.50]

# --- 3. Create the 3D Plot ---
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter points
ax.scatter(rocks_x, rocks_y, rocks_z, color='red', label='Rocks', marker='o', s=60)
ax.scatter(sand_x, sand_y, sand_z, color='blue', label='Sand', marker='x', s=60)

# Labeling
ax.set_title("3D Mars Terrain Feature Space")
ax.set_xlabel("Contrast (Roughness)")
ax.set_ylabel("Homogeneity (Smoothness)")
ax.set_zlabel("Energy (Uniformity)")

ax.legend()
plt.show()

ValueError: Key backend: 'module://ipympl.backend_nbagg' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template', 'inline']

In [ ]:
import plotly.graph_objects as go
import numpy as np

# --- 1. Your Actual Mars Data ---
rocks_x = [0.6965263157894737, 0.46247040802116696, 0.34945054945054943, 0.2659377316530763, 0.6124686716791979]
rocks_y = [0.776180185758514, 0.8267227934954042, 0.8566593406593406, 0.8768161601186063, 0.7510506834312672]
# Placeholder Energy (Replace with your actual Energy list)
rocks_z = [0.4505137971699016, 0.46822590170918854, 0.6266430305548975, 0.6000364537714598, 0.40207540958976895]

sand_x = [0.23267071124306182, 0.19218679838347552, 0.16777054333183655, 0.0275546422566785, 0.2084650509308044]
sand_y = [0.8899380405318188, 0.9043780871127076, 0.919280422092501, 0.9865589389127591, 0.9073586231120481]
# Placeholder Energy (Replace with your actual Energy list)
sand_z = [0.6163294697102372, 0.7764510077668599, 0.7925027055055208, 0.9725452537734324, 0.7858699529685078]

# --- 2. Build the Interactive Plot ---
fig = go.Figure()

# Add Rocks
fig.add_trace(go.Scatter3d(
    x=rocks_x, y=rocks_y, z=rocks_z,
    mode='markers',
    name='Rocks',
    marker=dict(size=8, color='red', symbol='circle', opacity=0.8)
))

# Add Sand
fig.add_trace(go.Scatter3d(
    x=sand_x, y=sand_y, z=sand_z,
    mode='markers',
    name='Sand',
    marker=dict(size=8, color='blue', symbol='x', opacity=0.8)
))

# --- 3. Labels and Layout ---
fig.update_layout(
    title="3D Mars Terrain Feature Map (Interactive)",
    scene = dict(
        xaxis_title='Contrast (Roughness)',
        yaxis_title='Homogeneity (Smoothness)',
        zaxis_title='Energy (Uniformity)'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# 1. Combine your data into a single feature matrix (X)
# Each row is an image: [Contrast, Homogeneity, Energy]
rocks = np.column_stack((rocks_x, rocks_y, rocks_z))
sand = np.column_stack((sand_x, sand_y, sand_z))

X = np.vstack((rocks, sand))

# 2. Create the labels (y)
y = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

# 3. Train the model (This is where the weights are calculated)
model = LogisticRegression()
model.fit(X, y)

# 4. Extract the weights and bias
weights = model.coef_[0]
bias = model.intercept_[0]

print(f"Weight for Contrast:    {weights[0]:.4f}")
print(f"Weight for Homogeneity: {weights[1]:.4f}")
print(f"Weight for Energy:      {weights[2]:.4f}")
print(f"Bias (b):               {bias:.4f}")

Weight for Contrast:    0.6605
Weight for Homogeneity: -0.2206
Weight for Energy:      -0.5964
Bias (b):               0.3666
